# Tree models: random forest and XGBoost

This notebook tunes a random forest and an XGBoost classifier to predict self-reported short sleep (fewer than seven hours per 24 hours) in BRFSS 2022. Results describe associations, not causes.

Scope of this notebook:
- Same encoded data (`brfss_2022_encoded.csv`), features, and train/validation/test assignments as the logistic regression baseline.
- Each model is tuned with a randomized search of 20 configurations, fit on train and scored on validation.
- Primary metric: survey-weighted average precision (AP) on validation.
- The test set is **not** used here. It is evaluated once, after the logistic regression is also tuned and the final model comparison is fixed.

In [8]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import xgboost
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)

SEED = 42
print(f"scikit-learn {sklearn.__version__}, xgboost {xgboost.__version__}, pandas {pd.__version__}")

scikit-learn 1.7.2, xgboost 3.4.2, pandas 2.2.3


## 1. Load the encoded data

We read `brfss_2022_encoded.csv`, exported by section 7 of the data preparation notebook. The logistic regression baseline reads the same file. It already represents the ten predictors as 79 numeric columns:

| Encoding | Predictors | Columns |
|---|---|---|
| Ordinal | `age_group`, `income_group`, `education_level`, `children_group` | One rank column (1 to K) plus an `_unknown` indicator. Unknown has rank 0. |
| One-hot | `sex`, `employment_status`, `marital_status`, `physical_activity`, `smoking_status`, `state` | One indicator per category, with one reference category dropped |

The mappings come from the CDC codebook, not from the data, so the same encoding applies to every partition without leaking validation or test information. The models therefore use these columns directly, with no encoder in the pipeline. For consistency, all models share the same encoded features, so differences in performance come from the models, not the inputs.

Ordinal codes suit trees well. A tree splits on thresholds, so it only uses the order of the ranks, not the spacing between them. It does not need the equal-step assumption that logistic regression makes. An Unknown rank of 0 sits below every known level, and a tree can separate it with one split or with the `_unknown` indicator.

`enough_sleep` is 1 for at least seven hours, so we invert it to make short sleep the positive class. The outcome, survey weight, design IDs (`stratum_id`, `psu_id`), and `split` are not features. Survey weights are passed separately as `sample_weight`.

`feature_groups` maps each original predictor to its encoded columns. Grouped permutation importance uses it later to shuffle all of a predictor's columns together.

In [9]:
cwd = Path.cwd()
repo_root = next(
    (
        folder
        for folder in (cwd, *cwd.parents)
        if (folder / "Data_preprocessing" / "brfss_2022_encoded.csv").is_file()
    ),
    None,
)
if repo_root is None:
    raise FileNotFoundError(
        "Could not find Data_preprocessing/brfss_2022_encoded.csv. "
        "Open the repository root in VS Code, or run the encoding section of the data preparation notebook first."
    )

DATA_FILE = repo_root / "Data_preprocessing" / "brfss_2022_encoded.csv"
RESULTS_DIR = repo_root / "Modeling" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

predictors = [
    "age_group", "sex", "income_group", "education_level",
    "employment_status", "marital_status", "children_group",
    "physical_activity", "smoking_status", "state",
]
metadata_columns = ["enough_sleep", "survey_weight", "stratum_id", "psu_id", "split"]

df = pd.read_csv(DATA_FILE)

# Each encoded column belongs to exactly one original predictor.
feature_groups = {
    predictor: [
        column for column in df.columns
        if column == predictor or column.startswith(predictor + "_")
    ]
    for predictor in predictors
}
features = [column for group in feature_groups.values() for column in group]

assert set(metadata_columns).issubset(df.columns), "Missing outcome or survey fields."
assert len(features) == len(set(features)), "An encoded column matched two predictors."
assert set(df.columns) == set(features) | set(metadata_columns), "Unexpected encoded CSV columns."
assert "sleep_hours" not in df.columns, "Sleep hours defines the outcome and must not be a feature."
assert all(pd.api.types.is_numeric_dtype(df[column]) for column in features)
assert np.isfinite(df[features].to_numpy()).all(), "Encoded predictors must be finite."
assert df["enough_sleep"].isin([0, 1]).all()
assert df["survey_weight"].gt(0).all()
df["short_sleep"] = 1 - df["enough_sleep"]

print(f"Respondents: {len(df):,}")
print(f"Short-sleep respondents: {df['short_sleep'].sum():,} ({df['short_sleep'].mean():.1%})")
print(f"Encoded features: {len(features)}")
display(
    pd.DataFrame(
        {
            "encoded columns": {p: len(cols) for p, cols in feature_groups.items()},
            "example": {p: ", ".join(cols[:3]) + (", ..." if len(cols) > 3 else "") for p, cols in feature_groups.items()},
        }
    )
)

Respondents: 427,573
Short-sleep respondents: 139,673 (32.7%)
Encoded features: 79


,encoded columns,example
age_group,2,"age_group, age_group_unknown"
sex,2,"sex_2, sex_Unknown"
income_group,2,"income_group, income_group_unknown"
education_level,2,"education_level, education_level_unknown"
employment_status,8,"employment_status_2, employment_status_3, empl..."
marital_status,6,"marital_status_2, marital_status_3, marital_st..."
children_group,2,"children_group, children_group_unknown"
physical_activity,1,physical_activity_2
smoking_status,4,"smoking_status_former, smoking_status_sometime..."
state,50,"state_AL, state_AR, state_AZ, ..."


## 2. Train, validation, and test partitions

We use the `split` column from the data preparation notebook without changing it. That notebook shuffled state-nested primary sampling units (state × PSU) with seed 42 and assigned 60/20/20 of the units to train, validation, and test. Each sampling unit appears in only one partition, so respondents from the same sampling cluster never sit on both sides of a split. The logistic regression baseline uses the same assignments, so scores are directly comparable.

**Survey weights.** Every partition's weights are divided by the mean *training* weight. Training weights then average one. Validation and test weights keep their relative sizes, and no statistic is computed from validation or test data. Rescaling all weights by a constant does not change weighted AP, ROC AUC, precision, recall, F1, or accuracy. It only puts the weighted confusion matrices on a scale comparable to the raw counts.

In [10]:
split_order = ["train", "validation", "test"]
assert set(df["split"]) == set(split_order), "Each respondent must have a valid split label."

train_weight_mean = df.loc[df["split"].eq("train"), "survey_weight"].mean()

X, y, w = {}, {}, {}
for name in split_order:
    part = df.loc[df["split"].eq(name)]
    X[name] = part[features]
    y[name] = part["short_sleep"].to_numpy()
    w[name] = (part["survey_weight"] / train_weight_mean).to_numpy()

assert np.isclose(w["train"].mean(), 1.0)

split_summary = pd.DataFrame(
    {
        "respondents": [len(y[s]) for s in split_order],
        "short sleep (sample %)": [100 * y[s].mean() for s in split_order],
        "short sleep (weighted %)": [100 * np.average(y[s], weights=w[s]) for s in split_order],
        "mean weight": [w[s].mean() for s in split_order],
    },
    index=split_order,
)
display(split_summary.round(3))

,respondents,short sleep (sample %),short sleep (weighted %),mean weight
train,256605,32.623,36.063,1.000
validation,85485,32.871,36.308,1.002
test,85483,32.591,35.743,1.023


## 3. Shared evaluation functions

These functions work with **any fitted model that has `predict_proba` and accepts the encoded feature DataFrame**. That includes the random forest and XGBoost models in this notebook, the logistic regression baseline, and a majority-class baseline.

- `evaluate` scores one partition. It returns survey-weighted and unweighted AP, ROC AUC, precision, recall, F1, and accuracy at a given threshold, plus a raw-count confusion matrix and a survey-weighted one.
- `pick_f1_threshold` picks the threshold that maximizes survey-weighted F1. It should only be called on the **validation** set. The chosen threshold is then applied unchanged to the test set.

AP and ROC AUC do not depend on the threshold. Precision, recall, F1, accuracy, and the confusion matrices do.

In [11]:
def predict_scores(model, X_part):
    """Predicted probability of short sleep (class 1)."""
    return model.predict_proba(X_part)[:, 1]


def evaluate(model, X_part, y_part, w_part, threshold=0.5, name=None):
    """Score a fitted model on one partition.

    Returns a dict with a flat `metrics` row and two labeled confusion matrices.
    """
    scores = predict_scores(model, X_part)
    pred = (scores >= threshold).astype(int)

    metrics = {"model": name or type(model).__name__, "threshold": threshold}
    for label, weights in [("weighted", w_part), ("unweighted", None)]:
        metrics[f"AP ({label})"] = average_precision_score(y_part, scores, sample_weight=weights)
        metrics[f"ROC AUC ({label})"] = roc_auc_score(y_part, scores, sample_weight=weights)
        metrics[f"Precision ({label})"] = precision_score(y_part, pred, sample_weight=weights, zero_division=0)
        metrics[f"Recall ({label})"] = recall_score(y_part, pred, sample_weight=weights, zero_division=0)
        metrics[f"F1 ({label})"] = f1_score(y_part, pred, sample_weight=weights, zero_division=0)
        metrics[f"Accuracy ({label})"] = accuracy_score(y_part, pred, sample_weight=weights)

    def labeled(cm):
        return pd.DataFrame(
            cm,
            index=["actual: enough sleep (0)", "actual: short sleep (1)"],
            columns=["predicted 0", "predicted 1"],
        )

    return {
        "metrics": metrics,
        "confusion_raw": labeled(confusion_matrix(y_part, pred, labels=[0, 1])),
        "confusion_weighted": labeled(confusion_matrix(y_part, pred, labels=[0, 1], sample_weight=w_part)),
    }


def pick_f1_threshold(model, X_val, y_val, w_val):
    """Threshold that maximizes survey-weighted F1. Use on validation only."""
    scores = predict_scores(model, X_val)
    precision, recall, thresholds = precision_recall_curve(y_val, scores, sample_weight=w_val)
    p, r = precision[:-1], recall[:-1]  # the last point has no threshold
    f1 = np.divide(2 * p * r, p + r, out=np.zeros_like(p), where=(p + r) > 0)
    best = int(np.argmax(f1))
    return float(thresholds[best]), float(f1[best])


def show_evaluation(result, digits=3):
    display(pd.DataFrame([result["metrics"]]).set_index("model").round(digits))
    print("Confusion matrix (raw counts):")
    display(result["confusion_raw"])
    print("Confusion matrix (survey-weighted):")
    display(result["confusion_weighted"].round(1))

### Smoke test of the evaluation functions

As a check on the functions only (this is **not** the reported majority-class baseline), we fit a constant-probability `DummyClassifier` on train and score it on validation. Every respondent gets the same score, so its weighted AP should equal the weighted short-sleep prevalence on validation, and its ROC AUC should be 0.5.

In [12]:
from sklearn.dummy import DummyClassifier

smoke_model = DummyClassifier(strategy="prior").fit(X["train"], y["train"], sample_weight=w["train"])
smoke = evaluate(smoke_model, X["validation"], y["validation"], w["validation"], name="smoke test (prior)")
show_evaluation(smoke)

val_prevalence = np.average(y["validation"], weights=w["validation"])
assert np.isclose(smoke["metrics"]["AP (weighted)"], val_prevalence)
assert np.isclose(smoke["metrics"]["ROC AUC (weighted)"], 0.5)
assert np.isclose(smoke["confusion_weighted"].to_numpy().sum(), w["validation"].sum())
print(f"Weighted validation prevalence: {val_prevalence:.3f} (matches the baseline notebook's constant-score AP of 0.363)")

,threshold,AP (weighted),ROC AUC (weighted),Precision (weighted),Recall (weighted),F1 (weighted),Accuracy (weighted),AP (unweighted),ROC AUC (unweighted),Precision (unweighted),Recall (unweighted),F1 (unweighted),Accuracy (unweighted)
model,,,,,,,,,,,,,
smoke test (prior),0.5,0.363,0.5,0.0,0.0,0.0,0.637,0.329,0.5,0.0,0.0,0.0,0.671


Confusion matrix (raw counts):


,predicted 0,predicted 1
actual: enough sleep (0),57385,0
actual: short sleep (1),28100,0


Confusion matrix (survey-weighted):


,predicted 0,predicted 1
actual: enough sleep (0),54547.5,0.0
actual: short sleep (1),31094.9,0.0


Weighted validation prevalence: 0.363 (matches the baseline notebook's constant-score AP of 0.363)


## 4. Random forest: randomized search

**Fixed settings.**
- `n_estimators=300`. More trees only make the averaged prediction more stable; they do not cause overfitting. 300 is enough for the average to settle, and it keeps each fit to about half a minute.
- `max_samples=0.5`. Each tree is trained on a random half of the training rows, drawn with replacement. This makes the trees more different from each other and halves the fitting time.
- `random_state=SEED`, so every run gives the same forests.

**Search space.** We draw 20 configurations from the grid below with `ParameterSampler`. Each one is fit on train with survey weights and scored on validation by weighted AP.

| Hyperparameter | Values | Why this range |
|---|---|---|
| `max_depth` | 6, 8, 10, 12, 16, 20, None | From shallow trees that only capture a few interactions up to fully grown trees |
| `min_samples_leaf` | 5, 10, 20, 50, 100, 200, 500 | Larger leaves average over more people, which smooths the predicted probabilities. With 256,605 training rows, even 500 is a small share of the data |
| `max_features` | `"sqrt"` (about 9 of 79), 0.2, 0.3, 0.5, 0.7 | Fewer features per split make the trees more different from each other; more features make each tree stronger |

We also record weighted AP on the training set. A large gap between train and validation AP signals overfitting. The validation AP alone decides which configuration we keep.

The logistic regression reference is the tuned baseline from `Baseline_model/logistic_baseline.ipynb` (`C=0.008`, validation weighted AP 0.454).

In [13]:
import time

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import ParameterSampler

LR_VAL_AP = 0.454  # tuned logistic regression baseline, validation weighted AP

RF_FIXED = {
    "n_estimators": 300,
    "max_samples": 0.5,
    "bootstrap": True,
    "n_jobs": -1,
    "random_state": SEED,
}
rf_space = {
    "max_depth": [6, 8, 10, 12, 16, 20, None],
    "min_samples_leaf": [5, 10, 20, 50, 100, 200, 500],
    "max_features": ["sqrt", 0.2, 0.3, 0.5, 0.7],
}
rf_configs = list(ParameterSampler(rf_space, n_iter=20, random_state=SEED))

rf_rows = []
rf_best_model, rf_best_params, rf_best_ap = None, None, -np.inf
for i, params in enumerate(rf_configs, start=1):
    start = time.perf_counter()
    model = RandomForestClassifier(**RF_FIXED, **params)
    model.fit(X["train"], y["train"], sample_weight=w["train"])

    val_scores = predict_scores(model, X["validation"])
    row = {
        "config": i,
        "max_depth": "None" if params["max_depth"] is None else params["max_depth"],
        "min_samples_leaf": params["min_samples_leaf"],
        "max_features": params["max_features"],
        "train AP (weighted)": average_precision_score(
            y["train"], predict_scores(model, X["train"]), sample_weight=w["train"]
        ),
        "val AP (weighted)": average_precision_score(y["validation"], val_scores, sample_weight=w["validation"]),
        "val ROC AUC (weighted)": roc_auc_score(y["validation"], val_scores, sample_weight=w["validation"]),
        "seconds": time.perf_counter() - start,
    }
    rf_rows.append(row)
    print(
        f"[{i:2d}/{len(rf_configs)}] depth={row['max_depth']!s:>4}, leaf={row['min_samples_leaf']:>3}, "
        f"features={row['max_features']!s:>4}: val AP {row['val AP (weighted)']:.4f} ({row['seconds']:.0f}s)"
    )

    # Keep only the best forest in memory; fully grown forests are large.
    if row["val AP (weighted)"] > rf_best_ap:
        rf_best_model, rf_best_params, rf_best_ap = model, params, row["val AP (weighted)"]
    del model

rf_search = pd.DataFrame(rf_rows)
rf_search["train - val AP"] = rf_search["train AP (weighted)"] - rf_search["val AP (weighted)"]
rf_search.to_csv(RESULTS_DIR / "rf_search.csv", index=False)
(RESULTS_DIR / "rf_best_params.json").write_text(json.dumps({**RF_FIXED, **rf_best_params}, indent=2))

[ 1/20] depth=   6, leaf= 50, features= 0.5: val AP 0.4542 (8s)
[ 2/20] depth=   6, leaf=500, features=sqrt: val AP 0.4562 (3s)
[ 3/20] depth=  16, leaf=500, features= 0.2: val AP 0.4619 (6s)
[ 4/20] depth=None, leaf= 20, features=sqrt: val AP 0.4683 (9s)
[ 5/20] depth=  20, leaf= 50, features= 0.5: val AP 0.4640 (20s)
[ 6/20] depth=  20, leaf= 20, features=sqrt: val AP 0.4664 (7s)
[ 7/20] depth=  20, leaf=100, features= 0.3: val AP 0.4655 (11s)
[ 8/20] depth=  12, leaf=200, features= 0.3: val AP 0.4636 (9s)
[ 9/20] depth=   6, leaf= 20, features= 0.2: val AP 0.4575 (4s)
[10/20] depth=  10, leaf= 50, features= 0.7: val AP 0.4617 (18s)
[11/20] depth=   8, leaf= 50, features= 0.2: val AP 0.4604 (5s)
[12/20] depth=None, leaf= 10, features=sqrt: val AP 0.4673 (11s)
[13/20] depth=  12, leaf=200, features= 0.2: val AP 0.4636 (7s)
[14/20] depth=  16, leaf=100, features= 0.7: val AP 0.4640 (27s)
[15/20] depth=  10, leaf=200, features= 0.2: val AP 0.4626 (7s)
[16/20] depth=  16, leaf=500, featu

179

In [14]:
display(
    rf_search.sort_values("val AP (weighted)", ascending=False)
    .set_index("config")
    .round({"train AP (weighted)": 4, "val AP (weighted)": 4, "val ROC AUC (weighted)": 4,
            "train - val AP": 4, "seconds": 0})
)
print(f"Best random forest: {rf_best_params}")
print(f"Validation weighted AP: {rf_best_ap:.4f} (logistic regression: {LR_VAL_AP:.3f}, "
      f"difference {rf_best_ap - LR_VAL_AP:+.4f})")

,max_depth,min_samples_leaf,max_features,train AP (weighted),val AP (weighted),val ROC AUC (weighted),seconds,train - val AP
config,,,,,,,,
4,None,20,sqrt,0.5462,0.4683,0.6182,9.0,0.0779
12,None,10,sqrt,0.6006,0.4673,0.6172,11.0,0.1333
6,20,20,sqrt,0.5286,0.4664,0.6175,7.0,0.0622
7,20,100,0.3,0.5060,0.4655,0.6169,11.0,0.0405
20,16,20,sqrt,0.5155,0.4652,0.6153,7.0,0.0503
17,20,10,0.3,0.6264,0.4644,0.6151,18.0,0.1620
14,16,100,0.7,0.5156,0.4640,0.6158,27.0,0.0516
5,20,50,0.5,0.5456,0.4640,0.6162,20.0,0.0817
8,12,200,0.3,0.4845,0.4636,0.6153,9.0,0.0209


Best random forest: {'min_samples_leaf': 20, 'max_features': 'sqrt', 'max_depth': None}
Validation weighted AP: 0.4683 (logistic regression: 0.454, difference +0.0143)


**Search results.**

The best configuration is fully grown trees (`max_depth=None`) with at least 20 people per leaf and `max_features="sqrt"`. It reaches a validation weighted AP of **0.468** and a weighted ROC AUC of **0.618**, compared with 0.454 and 0.602 for the tuned logistic regression. The gain of **+0.014 AP** is small but consistent: 19 of the 20 forests score above the logistic regression on validation, and the remaining one ties it.

- **Depth matters most.** The four forests with `max_depth=6` score 0.454 to 0.458, about the same as logistic regression. Every configuration in the top five has depth 16 or more. Deeper trees can combine several predictors in one path, so interactions seem to carry some of the signal.
- **Leaf size controls overfitting.** Larger leaves shrink the gap between train and validation AP. The gap falls from 0.133 with 10 people per leaf to 0.078 with 20 (both fully grown, `"sqrt"`), and validation AP is higher with 20. Very large leaves (200 to 500) nearly close the gap but lose validation AP, because the trees are too smooth to separate groups.
- **Fewer features per split work slightly better.** Four of the top five use `"sqrt"`, about 9 of 79 features. This keeps the trees more different from each other, so averaging them helps more.

The top five configurations lie within about 0.003 AP of each other, so the exact choice among them matters little. `"sqrt"` was the smallest `max_features` option in the search, so values below it were not tried.

The full search table is saved to `Modeling/results/rf_search.csv`, and the chosen settings to `Modeling/results/rf_best_params.json`. The best forest stays in memory as `rf_best_model` for the model comparison.

## 5. XGBoost: randomized search

XGBoost adds small trees one at a time. Each new tree fits the remaining errors of the current model and is added with a step size, the learning rate.

**Fixed settings.** `objective="binary:logistic"` gives predicted probabilities. `tree_method="hist"` bins each feature before splitting, which is fast on 256,605 rows. `random_state=SEED` makes the row and column sampling reproducible.

**Search space.** We draw 20 configurations with `ParameterSampler`, fit each on train with survey weights, and score it on validation by weighted AP. Continuous hyperparameters are drawn from distributions instead of a fixed grid.

| Hyperparameter | Distribution | Why this range |
|---|---|---|
| `max_depth` | 2, 3, 4, 5, 6 | Each tree only fixes part of the error, so shallow trees are enough. In a pilot fit, depth 8 already overfit |
| `learning_rate` | log-uniform, 0.01 to 0.3 | Log scale, because 0.01 vs 0.02 matters as much as 0.1 vs 0.2 |
| `n_estimators` | 100, 200, 300, 500, 800, 1200 | Trades off with the learning rate: a smaller step needs more trees |
| `subsample` | uniform, 0.5 to 1.0 | Share of rows each tree sees; less than 1 adds randomness |
| `colsample_bytree` | uniform, 0.4 to 1.0 | Share of the 79 features each tree sees |
| `reg_lambda` | log-uniform, 0.1 to 100 | L2 penalty on leaf values; larger values shrink each tree's contribution |

We do not use early stopping. It would pick the number of trees by watching the validation score during training, which would make validation part of the fitting. Instead, the number of trees is just another sampled hyperparameter, scored the same way as the others.

As with the random forest, we record weighted AP on train to check for overfitting, but only validation AP selects the configuration.

In [15]:
from scipy.stats import loguniform, uniform
from xgboost import XGBClassifier

XGB_FIXED = {
    "objective": "binary:logistic",
    "tree_method": "hist",
    "n_jobs": -1,
    "random_state": SEED,
}
xgb_space = {
    "max_depth": [2, 3, 4, 5, 6],
    "learning_rate": loguniform(0.01, 0.3),
    "n_estimators": [100, 200, 300, 500, 800, 1200],
    "subsample": uniform(0.5, 0.5),         # 0.5 to 1.0
    "colsample_bytree": uniform(0.4, 0.6),  # 0.4 to 1.0
    "reg_lambda": loguniform(0.1, 100),
}
# Convert NumPy scalars to plain Python numbers so they print and save cleanly.
xgb_configs = [
    {name: (int(value) if name in ("max_depth", "n_estimators") else float(value)) for name, value in params.items()}
    for params in ParameterSampler(xgb_space, n_iter=20, random_state=SEED)
]

xgb_rows = []
xgb_best_model, xgb_best_params, xgb_best_ap = None, None, -np.inf
for i, params in enumerate(xgb_configs, start=1):
    start = time.perf_counter()
    model = XGBClassifier(**XGB_FIXED, **params)
    model.fit(X["train"], y["train"], sample_weight=w["train"])

    val_scores = predict_scores(model, X["validation"])
    row = {
        "config": i,
        **params,
        "train AP (weighted)": average_precision_score(
            y["train"], predict_scores(model, X["train"]), sample_weight=w["train"]
        ),
        "val AP (weighted)": average_precision_score(y["validation"], val_scores, sample_weight=w["validation"]),
        "val ROC AUC (weighted)": roc_auc_score(y["validation"], val_scores, sample_weight=w["validation"]),
        "seconds": time.perf_counter() - start,
    }
    xgb_rows.append(row)
    print(
        f"[{i:2d}/{len(xgb_configs)}] depth={params['max_depth']}, lr={params['learning_rate']:.3f}, "
        f"trees={params['n_estimators']:>4}: val AP {row['val AP (weighted)']:.4f} ({row['seconds']:.0f}s)"
    )

    if row["val AP (weighted)"] > xgb_best_ap:
        xgb_best_model, xgb_best_params, xgb_best_ap = model, params, row["val AP (weighted)"]
    del model

xgb_search = pd.DataFrame(xgb_rows)
xgb_search["learning_rate x trees"] = xgb_search["learning_rate"] * xgb_search["n_estimators"]
xgb_search["train - val AP"] = xgb_search["train AP (weighted)"] - xgb_search["val AP (weighted)"]
xgb_search.to_csv(RESULTS_DIR / "xgb_search.csv", index=False)
(RESULTS_DIR / "xgb_best_params.json").write_text(json.dumps({**XGB_FIXED, **xgb_best_params}, indent=2))

[ 1/20] depth=4, lr=0.254, trees= 800: val AP 0.4473 (4s)
[ 2/20] depth=6, lr=0.048, trees= 500: val AP 0.4645 (3s)
[ 3/20] depth=3, lr=0.117, trees= 500: val AP 0.4673 (2s)
[ 4/20] depth=2, lr=0.043, trees= 300: val AP 0.4665 (1s)
[ 5/20] depth=4, lr=0.035, trees= 500: val AP 0.4711 (3s)
[ 6/20] depth=6, lr=0.079, trees= 100: val AP 0.4684 (1s)
[ 7/20] depth=2, lr=0.156, trees= 200: val AP 0.4696 (1s)
[ 8/20] depth=6, lr=0.015, trees= 300: val AP 0.4678 (2s)
[ 9/20] depth=3, lr=0.029, trees= 500: val AP 0.4684 (2s)
[10/20] depth=5, lr=0.244, trees=1200: val AP 0.4319 (7s)
[11/20] depth=6, lr=0.059, trees= 800: val AP 0.4532 (5s)
[12/20] depth=5, lr=0.026, trees= 800: val AP 0.4682 (4s)
[13/20] depth=2, lr=0.287, trees= 500: val AP 0.4691 (2s)
[14/20] depth=4, lr=0.112, trees= 100: val AP 0.4682 (1s)
[15/20] depth=2, lr=0.225, trees= 500: val AP 0.4689 (2s)
[16/20] depth=4, lr=0.030, trees= 500: val AP 0.4710 (3s)
[17/20] depth=4, lr=0.015, trees= 100: val AP 0.4554 (1s)
[18/20] depth=

293

In [16]:
display(
    xgb_search.sort_values("val AP (weighted)", ascending=False)
    .set_index("config")
    .round({"learning_rate": 4, "subsample": 3, "colsample_bytree": 3, "reg_lambda": 3,
            "train AP (weighted)": 4, "val AP (weighted)": 4, "val ROC AUC (weighted)": 4,
            "learning_rate x trees": 1, "train - val AP": 4, "seconds": 0})
)
print(f"Best XGBoost: { {k: round(v, 4) if isinstance(v, float) else v for k, v in xgb_best_params.items()} }")
print(f"Validation weighted AP: {xgb_best_ap:.4f} (logistic regression: {LR_VAL_AP:.3f}, "
      f"difference {xgb_best_ap - LR_VAL_AP:+.4f}; best random forest: {rf_best_ap:.4f})")

,colsample_bytree,learning_rate,max_depth,n_estimators,reg_lambda,subsample,train AP (weighted),val AP (weighted),val ROC AUC (weighted),seconds,learning_rate x trees,train - val AP
config,,,,,,,,,,,,
5,0.575,0.0348,4,500,3.489,0.796,0.5096,0.4711,0.6211,3.0,17.4,0.0385
16,0.587,0.0302,4,500,8.178,0.944,0.5030,0.4710,0.6206,3.0,15.1,0.0319
7,0.979,0.1564,2,200,0.196,0.842,0.4806,0.4696,0.6183,1.0,31.3,0.0110
13,0.445,0.2869,2,500,0.759,0.507,0.4940,0.4691,0.6195,2.0,143.5,0.0250
15,0.791,0.2246,2,500,0.983,0.532,0.4945,0.4689,0.6186,2.0,112.3,0.0256
9,0.798,0.0289,3,500,0.359,0.985,0.4862,0.4684,0.6181,2.0,14.4,0.0178
6,0.428,0.0790,6,100,0.157,0.974,0.5341,0.4684,0.6184,1.0,7.9,0.0657
14,0.519,0.1124,4,100,6.575,0.963,0.4936,0.4682,0.6185,1.0,11.2,0.0254
12,0.614,0.0260,5,800,0.265,0.901,0.5535,0.4682,0.6192,4.0,20.8,0.0853


Best XGBoost: {'colsample_bytree': 0.5753, 'learning_rate': 0.0348, 'max_depth': 4, 'n_estimators': 500, 'reg_lambda': 3.489, 'subsample': 0.7962}
Validation weighted AP: 0.4711 (logistic regression: 0.454, difference +0.0171; best random forest: 0.4683)


**Search results.**

The best configuration uses trees of depth 4, a learning rate of 0.035, and 500 trees, with each tree seeing 80% of rows and 58% of features (`reg_lambda` = 3.5). It reaches a validation weighted AP of **0.471** and a weighted ROC AUC of **0.621**. That is **+0.017 AP** over the tuned logistic regression (0.454) and +0.003 over the best random forest (0.468). The second-best configuration is almost the same (depth 4, learning rate 0.030, 500 trees) and scores 0.4710, so this region of the search is reliably good, not a lucky draw.

- **Too many steps overfit.** The product of learning rate and number of trees measures how far the model moves in total. The four configurations below logistic regression on validation all combine depth 4 to 6 with a large product (47 to 293). Their train AP is 0.16 to 0.27 higher than their validation AP.
- **Too few steps underfit.** The two configurations with a product below 3 (configs 17 and 18) score 0.455, about the same as logistic regression. They stop before learning much.
- **Shallow trees are forgiving.** Depth-2 trees stay between 0.467 and 0.470 even with a product above 100, and their train–validation gap stays at or below 0.026. Each depth-2 tree only captures one interaction, so adding many of them overfits slowly.
- **`reg_lambda` shows no clear pattern** in this search. Configs 8 and 19 both use a large penalty (about 53) at depth 6, but config 19 still overfits with 500 trees. The penalty does not offset too many boosting steps.

The eleven best configurations lie within 0.004 AP of each other, so many settings work about equally well. The gap between XGBoost and the random forest (0.003) is of the same size. On validation the two tree models perform about the same, and both beat logistic regression. By our selection rule, XGBoost has the higher validation AP, so it is the tree model we carry into the test-set comparison. It also fits several times faster than the forest (about 2 seconds versus 9).

The search table is saved to `Modeling/results/xgb_search.csv`, and the chosen settings to `Modeling/results/xgb_best_params.json`. The best model stays in memory as `xgb_best_model`.